# 🛠️ Notebook 2: ATM — Implementation

We'll build the ATM from Notebook 1 end-to-end. The code is intentionally small and commented so you can follow every line.

**Features implemented**
- Card + PIN authentication with a **3-tries-then-block** rule.
- A proper **state machine** (`IDLE → CARD_INSERTED → AUTHENTICATED → TRANSACTING`).
- Four transactions as separate classes: `BalanceInquiry`, `Deposit`, `Withdraw`, `Transfer` (**Command pattern**).
- Checking **and** savings accounts.
- **Daily withdrawal limit** per account.
- A **cash dispenser** that runs out of money.
- A receipt **printer** and an internal **transaction log**.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/atm
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1) Domain: Bank, Customer, Account, Card

In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import date
from enum import Enum
from typing import Dict, List, Optional


class AccountType(Enum):
    CHECKING = "checking"
    SAVINGS  = "savings"


@dataclass
class Account:
    id: str
    kind: AccountType
    balance: float = 0.0
    daily_withdraw_limit: float = 400.0
    # date -> amount already withdrawn that day
    _withdrawn_today: Dict[date, float] = field(default_factory=dict)

    # `balance` is the account's own money. Every path that changes it goes
    # through one of these three methods, so "never positive-amount-free" and
    # "never overdrawn" are enforced in exactly one place. Compare that with
    # letting callers write `account.balance -= x` from wherever they like:
    # then the rule lives in every caller, which means it lives nowhere.
    def debit(self, amount: float) -> None:
        """Take money out. No daily cap — that rule belongs to *cash* only."""
        if amount <= 0:
            raise ValueError("amount must be positive")
        if amount > self.balance:
            raise RuntimeError("insufficient funds")
        self.balance -= amount

    def withdraw(self, amount: float, today: date) -> None:
        """Cash out of a machine: `debit` plus the daily cash limit."""
        if amount <= 0:
            raise ValueError("amount must be positive")
        used = self._withdrawn_today.get(today, 0.0)
        if used + amount > self.daily_withdraw_limit:
            raise RuntimeError(
                f"daily limit ${self.daily_withdraw_limit:.0f} exceeded "
                f"(already took ${used:.0f})"
            )
        self.debit(amount)                    # may still raise "insufficient funds"
        self._withdrawn_today[today] = used + amount

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("amount must be positive")
        self.balance += amount


@dataclass
class Customer:
    name: str
    accounts: Dict[AccountType, Account]


@dataclass
class Card:
    number: str
    pin: str
    customer: Customer
    blocked: bool = False


class Bank:
    """The bank owns customers, cards and the rule 'block after 3 bad PINs'.

    In the real world this runs on bank servers and the ATM talks to it over
    the network. Here it's a simple in-memory object so the notebook runs
    without any external services.
    """

    MAX_PIN_TRIES = 3

    def __init__(self):
        self._cards: Dict[str, Card] = {}
        self._bad_tries: Dict[str, int] = {}

    def register(self, card: Card) -> None:
        self._cards[card.number] = card
        self._bad_tries[card.number] = 0

    def authenticate(self, card_number: str, pin: str) -> Card:
        card = self._cards[card_number]
        if card.blocked:
            raise RuntimeError("card blocked — contact your bank")
        if card.pin != pin:
            self._bad_tries[card_number] += 1
            if self._bad_tries[card_number] >= self.MAX_PIN_TRIES:
                card.blocked = True
                raise RuntimeError("too many wrong PINs — card blocked")
            raise RuntimeError(
                f"wrong PIN ({self._bad_tries[card_number]}/{self.MAX_PIN_TRIES})"
            )
        self._bad_tries[card_number] = 0  # reset on success
        return card


## 2) Hardware abstractions

One class per device. Makes the ATM easy to test with fakes.


In [ ]:
class CashDispenser:
    def __init__(self, cash: float):
        self.cash = cash

    def dispense(self, amount: float) -> None:
        if amount > self.cash:
            raise RuntimeError("ATM cannot dispense — not enough cash in machine")
        self.cash -= amount


class DepositSlot:
    def __init__(self):
        self.collected = 0.0

    def accept(self, amount: float) -> None:
        self.collected += amount


class Screen:
    def show(self, msg: str) -> None:
        print("🖥️  ", msg)


class Printer:
    def print_receipt(self, lines: List[str]) -> None:
        print("🧾 RECEIPT")
        for line in lines:
            print("    " + line)
        print("   -----")


## 3) Transactions — the Command pattern

Each transaction is a small class that knows how to `execute` itself against the ATM. Adding a new one (say, `PayBill`) means adding a new class — no existing code changes.


In [ ]:
class Transaction(ABC):
    @abstractmethod
    def execute(self, atm: "ATM") -> None: ...


class BalanceInquiry(Transaction):
    def __init__(self, account: Account):
        self.account = account

    def execute(self, atm):
        atm.screen.show(f"{self.account.kind.value} balance: ${self.account.balance:.2f}")
        atm.printer.print_receipt([
            f"Balance inquiry on {self.account.id}",
            f"Balance: ${self.account.balance:.2f}",
        ])
        atm.log.append(("balance", self.account.id, self.account.balance))


class Deposit(Transaction):
    def __init__(self, account: Account, amount: float):
        self.account = account
        self.amount = amount

    def execute(self, atm):
        atm.deposit_slot.accept(self.amount)
        self.account.deposit(self.amount)
        atm.screen.show(f"Deposited ${self.amount:.2f}")
        atm.printer.print_receipt([
            f"Deposit to {self.account.id}",
            f"Amount: ${self.amount:.2f}",
            f"New balance: ${self.account.balance:.2f}",
        ])
        atm.log.append(("deposit", self.account.id, self.amount))


class Withdraw(Transaction):
    def __init__(self, account: Account, amount: float):
        self.account = account
        self.amount = amount

    def execute(self, atm):
        # Check dispenser FIRST so we don't debit the account then fail.
        if self.amount > atm.dispenser.cash:
            raise RuntimeError("ATM cannot dispense — not enough cash in machine")
        self.account.withdraw(self.amount, atm.today)
        atm.dispenser.dispense(self.amount)
        atm.screen.show(f"Please take ${self.amount:.2f}")
        atm.printer.print_receipt([
            f"Withdraw from {self.account.id}",
            f"Amount: ${self.amount:.2f}",
            f"New balance: ${self.account.balance:.2f}",
        ])
        atm.log.append(("withdraw", self.account.id, self.amount))


class Transfer(Transaction):
    def __init__(self, source: Account, target: Account, amount: float):
        self.source, self.target, self.amount = source, target, amount

    def execute(self, atm):
        if self.source is self.target:
            raise ValueError("cannot transfer to the same account")
        # `debit`, not `withdraw`: the daily *cash* limit applies to notes
        # leaving the machine, not to money moving between your own accounts.
        # And not `source.balance -= amount` either — the balance rule stays
        # inside Account, where it can be enforced.
        self.source.debit(self.amount)
        self.target.deposit(self.amount)
        atm.screen.show(
            f"Transferred ${self.amount:.2f} "
            f"{self.source.kind.value} → {self.target.kind.value}"
        )
        atm.printer.print_receipt([
            f"Transfer {self.source.id} → {self.target.id}",
            f"Amount: ${self.amount:.2f}",
        ])
        atm.log.append(("transfer", self.source.id, self.target.id, self.amount))


## 4) The ATM controller with a state machine

The `State` enum plus a guard at the top of every method is the whole pattern:
**a method may only run in the states where it makes sense.**

```
IDLE ──insert_card──▶ CARD_INSERTED ──enter_pin──▶ AUTHENTICATED ⇄ TRANSACTING
  ▲                        │ 3 wrong PINs                │
  └──────── eject ─────────┴────────── eject ────────────┘
```

`TRANSACTING` is the state most people leave out of the diagram, and it is the
one that catches real bugs: while the dispenser is counting notes the machine
must refuse a *second* transaction on the same authorisation. Note the
`try/finally` — a state machine that can get stuck on an exception is worse
than no state machine, because the ATM would swallow the card and stop
responding.

In [ ]:
class State(Enum):
    IDLE          = "idle"
    CARD_INSERTED = "card_inserted"
    AUTHENTICATED = "authenticated"
    TRANSACTING   = "transacting"      # busy: dispensing cash, printing, …


class ATM:
    def __init__(self, bank: Bank, dispenser: CashDispenser,
                 deposit_slot: DepositSlot, screen: Screen, printer: Printer,
                 today: Optional[date] = None):
        self.bank = bank
        self.dispenser = dispenser
        self.deposit_slot = deposit_slot
        self.screen = screen
        self.printer = printer
        self.today = today or date.today()

        self.state: State = State.IDLE
        self.card: Optional[Card] = None
        self.log: list = []

    # ---- state transitions ------------------------------------------------
    def insert_card(self, card: Card) -> None:
        if self.state != State.IDLE:
            raise RuntimeError("ATM is busy")
        if card.blocked:
            raise RuntimeError("card blocked — keeping card")
        self.card = card
        self.state = State.CARD_INSERTED
        self.screen.show("Card accepted. Please enter PIN.")

    def enter_pin(self, pin: str) -> None:
        if self.state != State.CARD_INSERTED:
            raise RuntimeError("insert card first")
        try:
            self.bank.authenticate(self.card.number, pin)
        except RuntimeError as err:
            self.screen.show(str(err))
            # If the bank just blocked the card, eject and reset.
            if self.card.blocked:
                self.eject()
            raise
        self.state = State.AUTHENTICATED
        self.screen.show(f"Welcome, {self.card.customer.name}!")

    def run(self, transaction: Transaction) -> None:
        if self.state != State.AUTHENTICATED:
            raise RuntimeError("not authenticated")
        # TRANSACTING is not decoration. While the dispenser is counting notes
        # the machine must refuse a second transaction — otherwise a re-entrant
        # call (a double-tap on the screen, a retrying client) runs two
        # withdrawals against one authorisation. `finally` guarantees we come
        # back to AUTHENTICATED even when the transaction raises, so a failed
        # withdrawal does not wedge the ATM.
        self.state = State.TRANSACTING
        try:
            transaction.execute(self)
        finally:
            self.state = State.AUTHENTICATED

    def eject(self) -> None:
        if self.state is State.TRANSACTING:
            raise RuntimeError("cannot eject mid-transaction")
        self.card = None
        self.state = State.IDLE
        self.screen.show("Card ejected. Have a nice day!")


## 5) Happy-path demo

In [ ]:
from datetime import date

# --- set up the bank -------------------------------------------------------
alice = Customer(
    name="Alice",
    accounts={
        AccountType.CHECKING: Account("A-CHK", AccountType.CHECKING, balance=500.0),
        AccountType.SAVINGS:  Account("A-SAV", AccountType.SAVINGS,  balance=1500.0),
    },
)
card = Card(number="CARD-1", pin="1234", customer=alice)

bank = Bank()
bank.register(card)

# --- set up the ATM --------------------------------------------------------
atm = ATM(
    bank=bank,
    dispenser=CashDispenser(cash=1000.0),
    deposit_slot=DepositSlot(),
    screen=Screen(),
    printer=Printer(),
    today=date(2026, 4, 20),
)

# --- run a session ---------------------------------------------------------
atm.insert_card(card)
atm.enter_pin("1234")

chk = alice.accounts[AccountType.CHECKING]
sav = alice.accounts[AccountType.SAVINGS]

atm.run(BalanceInquiry(chk))
atm.run(Withdraw(chk, 200))           # cash out
atm.run(Deposit(sav, 100))            # deposit cash into savings
atm.run(Transfer(sav, chk, 300))      # internal transfer
atm.run(BalanceInquiry(chk))
atm.run(BalanceInquiry(sav))

atm.eject()
print("\nATM cash left:", atm.dispenser.cash)
print("Deposits collected:", atm.deposit_slot.collected)
print("Transaction log:", atm.log)


## 6) Error paths — the interesting bits

These are the edge cases a real ATM has to handle. We **prove** each rule by triggering it.


In [ ]:
# -- wrong PIN three times blocks the card ---------------------------------
bob = Customer("Bob", {AccountType.CHECKING: Account("B-CHK", AccountType.CHECKING, 200)})
bob_card = Card("CARD-2", "4321", bob)
bank.register(bob_card)

atm.insert_card(bob_card)
for attempt in range(3):
    try:
        atm.enter_pin("0000")
    except RuntimeError as e:
        print(f"attempt {attempt+1}: {e}")

print("card blocked?", bob_card.blocked)
print("ATM state after block:", atm.state)

# -- trying to use a blocked card is refused up front ----------------------
try:
    atm.insert_card(bob_card)
except RuntimeError as e:
    print("expected:", e)


In [ ]:
# -- cannot withdraw without authenticating --------------------------------
carol = Customer("Carol", {AccountType.CHECKING: Account("C-CHK", AccountType.CHECKING, 400)})
carol_card = Card("CARD-3", "1111", carol)
bank.register(carol_card)

atm.insert_card(carol_card)
try:
    atm.run(Withdraw(carol.accounts[AccountType.CHECKING], 50))
except RuntimeError as e:
    print("expected:", e)
atm.eject()


In [ ]:
# -- daily withdraw limit --------------------------------------------------
atm.insert_card(carol_card)
atm.enter_pin("1111")
chk_c = carol.accounts[AccountType.CHECKING]
chk_c.daily_withdraw_limit = 100  # small limit for demo

atm.run(Withdraw(chk_c, 80))
try:
    atm.run(Withdraw(chk_c, 50))  # 80 + 50 > 100
except RuntimeError as e:
    print("expected:", e)
atm.eject()


In [ ]:
# -- ATM physically out of cash --------------------------------------------
small_atm = ATM(bank, CashDispenser(cash=20.0), DepositSlot(), Screen(), Printer(),
                today=date(2026, 4, 20))
small_atm.insert_card(card)
small_atm.enter_pin("1234")
try:
    small_atm.run(Withdraw(chk, 100))  # account has money, machine does not
except RuntimeError as e:
    print("expected:", e)
print("account balance unchanged:", chk.balance)  # must be untouched
small_atm.eject()


In [ ]:
# -- the ATM is single-tracked: no second transaction mid-transaction -------
class SneakyDoubleWithdraw(Transaction):
    """Simulates a re-entrant call: a retry, a double-tap, a buggy client."""
    def __init__(self, account, amount):
        self.account, self.amount = account, amount
    def execute(self, atm):
        atm.screen.show(f"state during execute: {atm.state.value}")
        atm.run(Withdraw(self.account, self.amount))   # ← must be refused

dave = Customer("Dave", {AccountType.CHECKING: Account("D-CHK", AccountType.CHECKING, 900)})
dave_card = Card("CARD-4", "2222", dave)
bank.register(dave_card)

atm.insert_card(dave_card)
atm.enter_pin("2222")
try:
    atm.run(SneakyDoubleWithdraw(dave.accounts[AccountType.CHECKING], 100))
except RuntimeError as e:
    print("expected:", e)
print("balance untouched:", dave.accounts[AccountType.CHECKING].balance)
print("state recovered to:", atm.state.value)   # finally: → back to AUTHENTICATED
atm.eject()

## 7) Verify the design

The demos above *show* the rules; these assertions *pin them down*. Run this cell
after any change to the classes above — it is the cheapest regression suite there is.

In [ ]:
from datetime import date as _date

def scenario(balance=500.0, cash=1000.0, pin="1234"):
    """Build an isolated bank + ATM so assertions never share state."""
    cust = Customer("T", {
        AccountType.CHECKING: Account("T-CHK", AccountType.CHECKING, balance),
        AccountType.SAVINGS:  Account("T-SAV", AccountType.SAVINGS,  balance),
    })
    cd = Card("T-CARD", pin, cust)
    bk = Bank(); bk.register(cd)
    machine = ATM(bk, CashDispenser(cash), DepositSlot(), Screen(), Printer(),
                  today=_date(2026, 4, 20))
    return cust, cd, bk, machine

def rejects(fn, *a, **kw):
    """Assert that a call is refused, and report it if it isn't."""
    try:
        fn(*a, **kw)
    except (RuntimeError, ValueError):
        return True
    raise AssertionError(f"{getattr(fn, '__name__', fn)} should have been refused")

# ── State machine: every method is guarded ──────────────────────────────
cust, cd, bk, m = scenario()
chk = cust.accounts[AccountType.CHECKING]
assert m.state is State.IDLE
rejects(m.enter_pin, "1234")                      # no card yet
rejects(m.run, BalanceInquiry(chk))               # not authenticated
m.insert_card(cd)
rejects(m.insert_card, cd)                        # ATM already busy
rejects(m.run, BalanceInquiry(chk))               # PIN not entered
m.enter_pin("1234")
assert m.state is State.AUTHENTICATED
m.run(BalanceInquiry(chk))
assert m.state is State.AUTHENTICATED, "run() must restore the state it borrowed"
m.eject()
assert m.state is State.IDLE and m.card is None

# ── Money is conserved: no transaction creates or destroys value ────────
cust, cd, bk, m = scenario(balance=500.0)
chk, sav = cust.accounts[AccountType.CHECKING], cust.accounts[AccountType.SAVINGS]
before = chk.balance + sav.balance
m.insert_card(cd); m.enter_pin("1234")
m.run(Transfer(sav, chk, 200))
assert chk.balance + sav.balance == before, "a transfer must be value-neutral"
assert chk.balance == 700 and sav.balance == 300
m.run(Deposit(chk, 50))
assert chk.balance == 750 and m.deposit_slot.collected == 50

# ── Account invariants, wherever the money is asked to move ─────────────
for bad in (0, -1):
    rejects(chk.debit, bad); rejects(chk.deposit, bad)
    rejects(chk.withdraw, bad, m.today)
rejects(chk.debit, 10_000)                        # overdraft
rejects(Transfer(chk, chk, 10).execute, m)        # self-transfer
rejects(Transfer(chk, sav, 10_000).execute, m)    # transfer over balance
assert chk.balance == 750, "a refused operation must not move any money"

# ── The cash limit is about CASH, not about transfers ───────────────────
chk.daily_withdraw_limit = 100
m.run(Withdraw(chk, 60))
rejects(m.run, Withdraw(chk, 60))                 # 60 + 60 > 100
m.run(Transfer(chk, sav, 500))                    # internal move: no cash cap
assert chk.balance == 750 - 60 - 500

# ── The machine's cash and the account balance are separate resources ───
cust, cd, bk, m = scenario(balance=500.0, cash=20.0)
chk = cust.accounts[AccountType.CHECKING]
m.insert_card(cd); m.enter_pin("1234")
rejects(m.run, Withdraw(chk, 100))                # rich customer, empty ATM
assert chk.balance == 500.0, "a failed dispense must not debit the account"
assert m.dispenser.cash == 20.0
assert m.state is State.AUTHENTICATED, "a failed transaction must not wedge the ATM"

# ── Three strikes, and the bank (not the ATM) blocks the card ───────────
cust, cd, bk, m = scenario(pin="1234")
m.insert_card(cd)
for _ in range(Bank.MAX_PIN_TRIES - 1):
    rejects(m.enter_pin, "0000")
    assert not cd.blocked and m.state is State.CARD_INSERTED
rejects(m.enter_pin, "0000")
assert cd.blocked and m.state is State.IDLE, "blocking must eject the card"
rejects(m.insert_card, cd)                        # blocked card refused up front
rejects(bk.authenticate, cd.number, "1234")       # even the right PIN is dead now

# ── A correct PIN clears the strike counter ────────────────────────────
cust, cd, bk, m = scenario(pin="1234")
m.insert_card(cd)
rejects(m.enter_pin, "0000")
m.enter_pin("1234")
assert bk._bad_tries[cd.number] == 0
m.eject()

# ── Command pattern: adding a transaction touches no existing class ────
class PayBill(Transaction):
    def __init__(self, account, payee, amount):
        self.account, self.payee, self.amount = account, payee, amount
    def execute(self, atm):
        self.account.debit(self.amount)
        atm.screen.show(f"Paid ${self.amount:.2f} to {self.payee}")
        atm.log.append(("paybill", self.account.id, self.payee, self.amount))

cust, cd, bk, m = scenario(balance=300.0)
chk = cust.accounts[AccountType.CHECKING]
m.insert_card(cd); m.enter_pin("1234")
m.run(PayBill(chk, "City Water", 40))             # ATM.run never changed
assert chk.balance == 260 and m.log[-1][0] == "paybill"
assert issubclass(PayBill, Transaction)

print("✅ state machine, money conservation, limits, blocking and Command all verified")

## 7) Try it yourself — extensions

Good practice: try to implement one of these on top of the code above.

1. **Receipt toggle**: let the user choose "no receipt" — the `ATM` shouldn't print one.
2. **Mini-statement**: a new `MiniStatement(Transaction)` that prints the last 5 entries from `atm.log` for the card's accounts.
3. **PayBill**: add a `PayBill(Transaction)` that sends money to an external payee. Notice how you don't need to modify `ATM` at all — that's the Open/Closed Principle paying off.
4. **Concurrency**: two ATMs share the same `Bank`. What happens if both withdraw from the same account at once? Add a `threading.Lock` on `Account` and test.
5. **Timeout**: if the user takes longer than N seconds between `enter_pin` and `run`, auto-eject.

### What you practiced
- **Single Responsibility** — hardware, bank and transactions each live in their own class.
- **Command pattern** — `Transaction` + its subclasses.
- **State machine** — the `State` enum + guards in `ATM` methods, including the
  `TRANSACTING` state that blocks re-entrant transactions and a `finally` that
  always returns the machine to a usable state.
- **Encapsulation** — `balance` only ever changes through `Account.debit` /
  `deposit` / `withdraw`, so "no overdrafts" is enforced in one place instead of
  being re-implemented by every caller.
- **Dependency injection** — the ATM receives its hardware and bank, so you can swap fakes in tests.
